# 0.1 Download do METAR de SBSP (REDEMET)
Coleta do METAR de SBSP -  Aeroporto de Congonhas e decodificação pelo parser **MetPy**.

## Fonte

- Portal: <https://redemet.decea.mil.br>
- API: <https://ajuda.decea.mil.br/base-de-conhecimento/api-redemet-o-que-e/>
- API METAR: <https://ajuda.decea.mil.br/base-de-conhecimento/api-redemet-mensagem-metar/>
- Cadastro API: <https://api-redemet.decea.mil.br/cadastro-api/>

## Exemplo de solicitação
- <https://api-redemet.decea.mil.br/mensagens/metar/SBSP?api_key=SUA_CHAVE_AQUI&data_ini=2025010100&data_fim=2025013124>

## Saída: 
- `data/processed/cgh_metar_2025.parquet`

In [1]:
# pip install metpy
# pip install pandas

In [2]:
import pandas as pd
import requests
from metpy.io import parse_metar_to_dataframe
from metpy.units import units

## 0.1.1 Download das mensagens

In [ ]:
airport = "SBSP"
page_tam = 200                      # Limitado a 200 pela REDEMET, valor padrão é 150 da API
date_initial = "01/01/2025 00:00"      # Data inicial no formato DD/MM/YYYY HH:MM em UTC (Aeroporto de São Paulo-Congonhas abre apartir das 09:00 UTC, ou seja, 06:00 horário de Brasília para a aviação comercial)
date_end = "01/01/2026 03:00"          # Data final no formato DD/MM/YYYY HH:MM em UTC (Aeroporto de São Paulo-Congonhas fecha apartir das 02:00 UTC+1, ou seja, 23:00 horário de Brasília para a aviação comercial)
BASE = "https://api-redemet.decea.mil.br/mensagens/metar"

# Cole aqui a sua chave de acesso da REDEMET antes de rodar.
API_KEY = "XXX"

In [267]:
# Observações: Existe um limite de 8.760 registros por chamada, na prática, esta limitado a 365 dias por ano cada chamada. 
# Por esta razao que dividi as chamadas anualmente.
date_initial_format = pd.to_datetime(date_initial, format='%d/%m/%Y %H:%M')
date_end_format = pd.to_datetime(date_end, format='%d/%m/%Y %H:%M')
value = date_end_format - date_initial_format

if value > pd.Timedelta(days=365):
    print("O intervalo de datas é maior ou igual a 1 ano.")
    linhas = []
    inicio = date_initial_format
    while inicio < date_end_format:
        fim = min(inicio + pd.Timedelta(days=365), date_end_format)
        linhas.append({
            'data_ini': inicio.strftime('%Y%m%d%H'),
            'data_fim': fim.strftime('%Y%m%d%H')
        })
        inicio = fim
    dates = pd.DataFrame(linhas)
else:
    print("O intervalo de datas é menor ou igual a 1 ano.")
    inicio = date_initial_format
    fim = date_end_format
    dates = pd.DataFrame({'data_ini': [inicio.strftime('%Y%m%d%H')], 'data_fim': [fim.strftime('%Y%m%d%H')]})

metar = pd.DataFrame()
 
for dates in dates.itertuples(index=False):
    page = 1
    data_ini = dates.data_ini
    data_fim = dates.data_fim
    url = f"https://api-redemet.decea.mil.br/mensagens/metar/{airport}?api_key={API_KEY}&page_tam={page_tam}&data_ini={data_ini}&data_fim={data_fim}&page={page}"
    response = requests.get(url)
    response.raise_for_status()
    body = response.json()
    if body['status'] == False:
        print(f"A REDEMET nao retornou dados. Resposta: {body['message']}.")
    else:
        metar_part = pd.DataFrame(body["data"]["data"])
        metar_part['validade_inicial'] = pd.to_datetime(metar_part['validade_inicial'], format='%Y-%m-%d %H:%M:%S')
        metar_part['recebimento'] = pd.to_datetime(metar_part['recebimento'], format='%Y-%m-%d %H:%M:%S')
        metar_part.set_index('validade_inicial', inplace=True)
        
        last_page = int(body["data"]["last_page"])
        metar = pd.concat([metar, metar_part], ignore_index=False)
        metar = metar[~metar.index.duplicated(keep="last")]

        print(f"Baixando dados do METAR para o aeroporto {airport} de {pd.to_datetime(data_ini, format='%Y%m%d%H').strftime('%d/%m/%Y %H:%M UTC')} a {pd.to_datetime(data_fim, format='%Y%m%d%H').strftime('%d/%m/%Y %H:%M UTC')}.")
        print(f"Total de páginas: {last_page}. ")
        print(f"Página {page} de {last_page}. Total de {len(metar)} registros.")
        
        while page < last_page:
            page = page + 1
            url = f"https://api-redemet.decea.mil.br/mensagens/metar/{airport}?api_key={API_KEY}&page_tam={page_tam}&data_ini={data_ini}&data_fim={data_fim}&page={page}"
            response = requests.get(url)
            response.raise_for_status()
            body = response.json()
            if body['status'] == False:
                print(f"A REDEMET nao retornou dados. Resposta: {body['message']}.")
                break
            else:
                metar_part = pd.DataFrame(body["data"]["data"])
                metar_part['validade_inicial'] = pd.to_datetime(metar_part['validade_inicial'], format='%Y-%m-%d %H:%M:%S')
                metar_part['recebimento'] = pd.to_datetime(metar_part['recebimento'], format='%Y-%m-%d %H:%M:%S')
                metar_part.set_index('validade_inicial', inplace=True)
                metar = pd.concat([metar, metar_part], ignore_index=False, sort=True)
                metar = metar[~metar.index.duplicated(keep="last")]

                print(f"Página {page} de {last_page}. Total de {len(metar)} registros.")

metar.sort_index(inplace=True)

O intervalo de datas é maior ou igual a 1 ano.
Baixando dados do METAR para o aeroporto SBSP de 01/01/2025 00:00 UTC a 01/01/2026 00:00 UTC.
Total de páginas: 58. 
Página 1 de 58. Total de 199 registros.
Página 2 de 58. Total de 399 registros.
Página 3 de 58. Total de 599 registros.
Página 4 de 58. Total de 799 registros.
Página 5 de 58. Total de 999 registros.
Página 6 de 58. Total de 1199 registros.
Página 7 de 58. Total de 1399 registros.
Página 8 de 58. Total de 1599 registros.
Página 9 de 58. Total de 1799 registros.
Página 10 de 58. Total de 1997 registros.
Página 11 de 58. Total de 2197 registros.
Página 12 de 58. Total de 2397 registros.
Página 13 de 58. Total de 2597 registros.
Página 14 de 58. Total de 2797 registros.
Página 15 de 58. Total de 2997 registros.
Página 16 de 58. Total de 3196 registros.
Página 17 de 58. Total de 3395 registros.
Página 18 de 58. Total de 3595 registros.
Página 19 de 58. Total de 3795 registros.
Página 20 de 58. Total de 3995 registros.
Página 21 

# 0.1.2 Metar Parser

In [271]:
metar.info()

<class 'pandas.DataFrame'>
DatetimeIndex: 11410 entries, 2025-01-01 00:00:00 to 2026-01-01 03:00:00
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   id_localidade  11410 non-null  str           
 1   mens           11410 non-null  str           
 2   recebimento    11410 non-null  datetime64[us]
dtypes: datetime64[us](1), str(2)
memory usage: 1.0 MB


In [272]:
metar_df = []
for idx, mens in zip(metar.index, metar["mens"]):
    metar_df_bloco = parse_metar_to_dataframe(mens)
    metar_df_bloco['mens'] = mens
    metar_df_bloco.index = [idx]
    metar_df.append(metar_df_bloco)
metar_df = pd.concat(metar_df)
# Metar Parser não calcula corretamente a data e hora do METAR, então vamos remover a coluna 'date_time' que é calculada incorretamente.
# Utilizamos a data do METAR que é a data do índice do DataFrame.
metar_df.drop('date_time', axis=1, inplace=True)

In [ ]:
CONVECTIVAS = r"(?:FEW|SCT|BKN|OVC|///)[\d/]{3}(CB|TCU)"
tipos = metar_df["mens"].str.extractall(CONVECTIVAS)[0]
metar_df["cb"] = metar_df.index.isin(
    tipos[tipos == "CB"].index.get_level_values(0)).astype(int)
metar_df["tcu"] = metar_df.index.isin(
    tipos[tipos == "TCU"].index.get_level_values(0)).astype(int)
# No dataframe original do Metar parser ele não fornece a pressão atmosferica na altitude local, então precisamos calcular manualmente.
metar_df["altimeter_hpa"] = (
    (metar_df["altimeter"].values * units.inHg).to("hPa").magnitude.round(0)
)

# 0.1.3 Gravação

In [296]:
metar_df.to_parquet("data/processed/cgh_metar_2025.parquet", index=False)
metar_df.to_csv("data/processed/cgh_metar_2025.csv", index=False)